# Module 11 Lab - Hyperparameter Tuning & AutoML

**Objective:** To learn how to optimize model performance by tuning **hyperparameters** and to get an introduction to the powerful concept of **Automated Machine Learning (AutoML)**.

**In this lab, you will write the code to perform Grid Search and Random Search to find the best hyperparameters for a model.**

## Part 1: What are Hyperparameters?

**Concept:** In machine learning, there are two types of parameters:

1.  **Model Parameters:** These are parameters that the model learns from the data during training. For example, the coefficients in a Linear Regression model.

2.  **Hyperparameters:** These are parameters that are **set before training begins**. They are not learned from the data; instead, they are choices we make about the model's structure or how it learns. 
    *   *Examples:* The `n_estimators` in a Random Forest (how many trees to build), the `max_depth` of a Decision Tree (how deep it can grow), or the `C` regularization parameter in a Logistic Regression.

Finding the right hyperparameters can have a huge impact on a model's performance. **Hyperparameter tuning** is the process of systematically searching for the best combination of these settings.

## Part 2: Setup

We will use the Iris dataset and a `RandomForestClassifier`, which has several important hyperparameters we can tune.

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load and prepare data
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# A baseline model with default hyperparameters
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"Accuracy of baseline Random Forest: {accuracy_baseline:.2%}")

Accuracy of baseline Random Forest: 100.00%


## Part 3: Grid Search

**Concept:** Grid Search is the most straightforward tuning method. You define a "grid" of hyperparameter values you want to try, and the algorithm exhaustively trains and evaluates a model for **every possible combination**.

*   **Pro:** It's guaranteed to find the best combination within the grid.
*   **Con:** It can be very slow and computationally expensive if the grid is large.

### Task 1: Perform a Grid Search

**Your Task:** Use `GridSearchCV` from `sklearn.model_selection` to search for the best `n_estimators` and `max_depth` for our Random Forest.

In [ ]:
from sklearn.model_selection import GridSearchCV

# --- ENTER YOUR CODE HERE ---

# 1. Define the grid of hyperparameters to search
#    This dictionary defines the 'grid'. It will test n_estimators=50, 100, 200 and max_depth=5, 10, None.
#    Total combinations: 3 * 3 = 9 models to train.
param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [5, 10, None]
# }

# 2. Create a GridSearchCV instance
#    `cv=5` means it will use 5-fold cross-validation for each combination.
#    `n_jobs=-1` tells it to use all available CPU cores to speed up the search.
# grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# 3. Fit the grid search to the data
# grid_search.fit(X_train, y_train)

# 4. Print the best parameters and the best score
# print(f"Best Parameters found by Grid Search: {grid_search.best_params_}")
# print(f"Best cross-validated score: {grid_search.best_score_:.2%}")

## Part 4: Random Search

**Concept:** Random Search is often more efficient than Grid Search. Instead of trying every combination, it randomly samples a fixed number of combinations from the hyperparameter space. 

*   **Pro:** It's much faster and can explore a wider range of values.
*   **Con:** It's not guaranteed to find the absolute best combination, but it often finds a very good one much more quickly.

### Task 2: Perform a Random Search

**Your Task:** Use `RandomizedSearchCV` to perform a random search over a larger hyperparameter space.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# --- ENTER YOUR CODE HERE ---

# 1. Define the distribution of hyperparameters to sample from
#    This is a larger space than we used for Grid Search.
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start = 50, stop = 500, num = 10)],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

# 2. Create a RandomizedSearchCV instance
#    `n_iter=10` means it will randomly sample and train 10 different combinations.
random_search = RandomizedSearchCV(estimator=RandomForestClassifier(random_state=42), param_distributions=param_dist, n_iter=10, cv=5, n_jobs=-1, verbose=2, random_state=42)

# 3. Fit the random search to the data
random_search.fit(X_train, y_train)

# 4. Print the best parameters and the best score
print(f"Best Parameters found by Random Search: {random_search.best_params_}")
print(f"Best cross-validated score: {random_search.best_score_:.2%}")

## Part 5: Introduction to AutoML with AutoGluon

**Concept:** AutoML takes hyperparameter tuning to the next level. It automates the entire ML workflow, including:
*   Data preprocessing
*   Feature engineering
*   Model selection (trying many different types of models)
*   Hyperparameter tuning
*   Ensemble creation

**AutoGluon** is a popular and easy-to-use AutoML library. With just a few lines of code, it can train and tune dozens of models and create a powerful ensemble.

**This part is fully coded.** Your task is to run it and see the power of AutoML. Note that it may take a few minutes to run.

In [4]:
# You may need to install AutoGluon first. Uncomment the line below in your Colab notebook.
# !pip install autogluon

from autogluon.tabular import TabularPredictor

# AutoGluon requires the data in a single DataFrame with the target column.
# We will create a training DataFrame for AutoGluon
train_data_ag = pd.DataFrame(X_train, columns=iris.feature_names)
train_data_ag['species'] = y_train

# Create a test DataFrame as well
test_data_ag = pd.DataFrame(X_test, columns=iris.feature_names)
test_data_ag['species'] = y_test

# Create and train the TabularPredictor
# `time_limit=60` tells it to run for 60 seconds.
predictor = TabularPredictor(label='species', eval_metric='accuracy').fit(train_data=train_data_ag, time_limit=60)

# Evaluate the predictor on the test data
leaderboard = predictor.leaderboard(test_data_ag)
print(leaderboard)

ModuleNotFoundError: No module named 'autogluon'